## Background
This Jupiter Notebook aims to set the classical baseline for manipulation classification task. \
At first data is preprocessed, including lowercasing, lemmatization, removal of emojis, punctuation, and URLs. Then texts are vectorized using TF-IDF and XLM-RoBERTa-large. Then 2 models will be tested: CatBoost and LightGBM.

## Imports

In [2]:
import os
import ast
import re
import warnings
import emoji
import pandas as pd
import numpy as np
from collections import Counter
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import classification_report

from transformers import XLMRobertaTokenizer, XLMRobertaModel
import torch
from tqdm import tqdm
import numpy as np

from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

import matplotlib.pyplot as plt

/Users/k_akhynko/Desktop/work/thesis/manipulative-narrative-detection/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
warnings.filterwarnings("ignore")

## Constants

In [4]:
TRAIN_PATH = "../../data/"
TRAIN_NAME = "train.parquet"

In [5]:
TEST_PATH = "../../data/"
TEST_NAME = "test_lang.csv"

## Read data

In [6]:
train = pd.read_parquet(os.path.join(TRAIN_PATH, TRAIN_NAME))
train.head()

,id,content,lang,manipulative,techniques,trigger_words
0,0bb0c7fa-101b-4583-a5f9-9d503339141c,Новий огляд мапи DeepState від російського вій...,uk,True,"[euphoria, loaded_language]","[[27, 63], [65, 88], [90, 183], [186, 308]]"
1,7159f802-6f99-4e9d-97bd-6f565a4a0fae,Недавно 95 квартал жёстко поглумился над русск...,ru,True,"[loaded_language, cherry_picking]","[[0, 40], [123, 137], [180, 251], [253, 274]]"
2,e6a427f1-211f-405f-bd8b-70798458d656,🤩\nТим часом йде евакуація Бєлгородського авто...,uk,True,"[loaded_language, euphoria]","[[55, 100]]"
3,1647a352-4cd3-40f6-bfa1-d87d42e34eea,В Україні найближчим часом мають намір посилит...,uk,False,None,None
4,9c01de00-841f-4b50-9407-104e9ffb03bf,"Расчёты 122-мм САУ 2С1 ""Гвоздика"" 132-й бригад...",ru,True,[loaded_language],"[[114, 144]]"


In [ ]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3822 entries, 0 to 3821
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   id             3822 non-null   object
 1   content        3822 non-null   object
 2   lang           3822 non-null   object
 3   manipulative   3822 non-null   bool  
 4   techniques     2589 non-null   object
 5   trigger_words  2589 non-null   object
dtypes: bool(1), object(5)
memory usage: 153.2+ KB


In [ ]:
# If there are no manipulations in the text, we will set techniques and trigger_words to empty lists
train['techniques'] = train['techniques'].apply(lambda x: [] if x is None else x)
train['trigger_words'] = train['trigger_words'].apply(lambda x: [] if x is None else x)

In [ ]:
# Number of occurrences of each technique
technique_counts = Counter([tech for sublist in train['techniques'] for tech in sublist])
technique_counts

Counter({'euphoria': 462,
         'loaded_language': 1973,
         'cherry_picking': 512,
         'glittering_generalities': 483,
         'cliche': 463,
         'appeal_to_fear': 300,
         'bandwagon': 157,
         'fud': 385,
         'whataboutism': 158,
         'straw_man': 138})

In [ ]:
test = pd.read_csv(os.path.join(TEST_PATH, TEST_NAME))
test.head()

,id,content,trigger_words,techniques,lang
0,521cd2e8-dd9f-42c4-98ba-c0c8890ff1ba,"Они просрали нашу технику, положили кучу людей...","[(0, 12), (27, 46), (48, 71), (131, 162), (164...","['fud', 'loaded_language']",ru
1,9b2a61e4-d14e-4ff7-b304-e73d720319bf,❗️\nКитай предлагает отдать оккупированные тер...,"[(374, 425)]",['loaded_language'],ru
2,f0f1c236-80a8-4d25-b30c-a420a39be632,Сегодня будет ровно 6 месяцев с этого обещания...,"[(0, 127)]",['loaded_language'],ru
3,31ea05ba-2c2b-4b84-aba7-f3cf6841b204,⚡️\nІзраїль вперше у світі збив балістичну рак...,[],[],uk
4,a79e13ec-6d9a-40b5-b54c-7f4f743a7525,Склав невелику навчально-методичну таблицю на ...,"[(87, 103), (127, 136), (170, 189), (204, 255)...",['loaded_language'],uk


In [ ]:
test['trigger_words'] = test['trigger_words'].apply(lambda x: x if isinstance(x, str) else "[]")

test['trigger_words'] = test['trigger_words'].apply(ast.literal_eval)
test['techniques'] = test['techniques'].apply(ast.literal_eval)

In [ ]:
Counter(test['lang'])

Counter({'ru': 2595, 'uk': 3131, 'et': 4, 'en': 2, 'bg': 3})

In [ ]:
test = test[test['lang'].isin(['uk', 'ru'])]

## Data preparation

### Spacy models setup

In [ ]:
spacy.prefer_gpu()

True

In [ ]:
# Download spacy models
!python -m spacy download uk_core_news_lg -q
!python -m spacy download ru_core_news_lg -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.2/231.2 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 73.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 83.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('uk_core_news_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 513.4/513.4 MB 1.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('ru_core_news_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do 

In [ ]:
# Load spaCy NER models
nlp_uk = spacy.load("uk_core_news_lg")
nlp_ru = spacy.load("ru_core_news_lg")

In [ ]:
# Define stopwords sets (from spaCy)
stopwords_uk = nlp_uk.Defaults.stop_words
stopwords_ru = nlp_ru.Defaults.stop_words

In [ ]:
# Define a dictionary for acronym expansion (Ukrainian & Russian)
acronym_dict = {
    "сша": "сполучені штати америки",
    "лол": "дуже смішно",
    "омг": "о боже",
    "бзв": "до вашого відома",
    "незнаю": "не знаю",  # Common typo fix

    "ссср": "союз советских социалистических республик",
    "нло": "неопознанный летающий объект",
    "мчс": "министерство чрезвычайных ситуаций",
}

### Preprocessing

In [ ]:
def preprocess_text(text, lang):
    if lang == "ru":
        nlp = nlp_ru
        stopwords_set = stopwords_ru
    else:
        nlp = nlp_uk
        stopwords_set = stopwords_uk

    # Convert to lowercase
    text = text.lower()

    # Expand acronyms
    for acronym, expanded in acronym_dict.items():
        text = re.sub(r'\b' + re.escape(acronym) + r'\b', expanded, text)

    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)

    # Remove punctuation
    text = re.sub(r'[^\w\s]', '', text)

    # Remove emojis
    text = emoji.replace_emoji(text, replace='')

    # Remove extra spaces and newlines
    text = re.sub(r'\s+', ' ', text)  # Replace multiple spaces with a single space
    text = text.replace('\n', ' ')    # Replace newline characters with a space
    text = text.strip()               # Remove leading/trailing spaces

    # Process text using spaCy (tokenization & lemmatization)
    doc = nlp(text)
    processed_words = [
        token.lemma_ for token in doc if token.text not in stopwords_set and not token.is_punct
    ]

    return ' '.join(processed_words)

In [22]:
train['clean_content'] = train.apply(lambda row: preprocess_text(row['content'], lang=row['lang']), axis=1)
test['clean_content'] = test.apply(lambda row: preprocess_text(row['content'], lang=row['lang']), axis=1)

### Data finalization

In [23]:
X_train_text = train['clean_content'].tolist()
X_test_text = test['clean_content'].tolist()

In [24]:
mlb = MultiLabelBinarizer()
y_train = mlb.fit_transform(train['techniques'])
y_test_true = mlb.transform(test['techniques'])

In [25]:
label_names = mlb.classes_
label_names

array(['appeal_to_fear', 'bandwagon', 'cherry_picking', 'cliche',
       'euphoria', 'fud', 'glittering_generalities', 'loaded_language',
       'straw_man', 'whataboutism'], dtype=object)

## Embeddings

In [26]:
def get_tfidf_features(train_texts, test_texts, max_features=10000):
    vectorizer = TfidfVectorizer(max_features=max_features)
    X_train = vectorizer.fit_transform(train_texts)
    X_test = vectorizer.transform(test_texts)
    return X_train, X_test

In [ ]:
def get_xlm_roberta_features(train_texts, test_texts, model_name='xlm-roberta-large'):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    tokenizer = XLMRobertaTokenizer.from_pretrained(model_name)
    model = XLMRobertaModel.from_pretrained(model_name).to(device)
    model.eval()

    def encode_texts(texts):
        features = []
        with torch.no_grad():
            for text in tqdm(texts):
                # Move input tensors to device
                inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True).to(device)
                outputs = model(**inputs)
                cls_embedding = outputs.last_hidden_state[:, 0, :]  # Take [CLS] token representation
                features.append(cls_embedding.squeeze().cpu().numpy())  # Move back to CPU
        return np.array(features)

    X_train = encode_texts(train_texts)
    X_test = encode_texts(test_texts)

    return X_train, X_test

# Modeling

In [28]:
def train_model(X_train, y_train, model_type='catboost'):
    if model_type == 'catboost':
        base_model = CatBoostClassifier(verbose=0, task_type="GPU", iterations=200, learning_rate=0.1)
    elif model_type == 'lightgbm':
        base_model = LGBMClassifier(n_estimators=200, learning_rate=0.1)

    model = MultiOutputClassifier(base_model)
    model.fit(X_train, y_train)
    return model

## Evaluation

In [29]:
def evaluate_model(model, X_test, y_test_true, label_names, description=""):
    y_pred = model.predict(X_test)
    print(f"\n Classification Report ({description}):")
    print(classification_report(y_test_true, y_pred, target_names=label_names))

In [30]:
# === TF-IDF + CatBoost ===
X_train_tfidf, X_test_tfidf = get_tfidf_features(X_train_text, X_test_text)
model_cb_tfidf = train_model(X_train_tfidf, y_train, model_type='catboost')

evaluate_model(model_cb_tfidf, X_test_tfidf, y_test_true, label_names, description='TF-IDF + CatBoost')


 Classification Report (TF-IDF + CatBoost):
                         precision    recall  f1-score   support

         appeal_to_fear       0.35      0.03      0.06       449
              bandwagon       0.00      0.00      0.00       236
         cherry_picking       0.48      0.10      0.17       768
                 cliche       0.40      0.01      0.01       694
               euphoria       0.55      0.09      0.15       694
                    fud       0.62      0.13      0.22       576
glittering_generalities       0.75      0.33      0.46       722
        loaded_language       0.66      0.72      0.69      2955
              straw_man       0.00      0.00      0.00       207
           whataboutism       0.00      0.00      0.00       235

              micro avg       0.65      0.34      0.45      7536
              macro avg       0.38      0.14      0.18      7536
           weighted avg       0.53      0.34      0.37      7536
            samples avg       0.39      0.2

In [31]:
# === TF-IDF + LightGBM ===
model_lgb_tfidf = train_model(X_train_tfidf, y_train, model_type='lightgbm')

evaluate_model(model_lgb_tfidf, X_test_tfidf, y_test_true, label_names, description='TF-IDF + LightGBM')

[LightGBM] [Info] Number of positive: 300, number of negative: 3522
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.070994 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 31139
[LightGBM] [Info] Number of data points in the train set: 3822, number of used features: 1710
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.078493 -> initscore=-2.463002
[LightGBM] [Info] Start training from score -2.463002
[LightGBM] [Info] Number of positive: 157, number of negative: 3665
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.037598 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 31139
[LightGBM] [Info] Number of data points in the train set: 3822, number of used features: 1710
[LightGBM] [Info] [b

In [35]:
# === XLM-RoBERTa + CatBoost ===
X_train_xlm, X_test_xlm = get_xlm_roberta_features(X_train_text, X_test_text)
model_cb_xlm = train_model(X_train_xlm, y_train, model_type='catboost')

evaluate_model(model_cb_xlm, X_test_xlm, y_test_true, label_names, description='XLM-RoBERTa + CatBoost')

100%|██████████| 5726/5726 [02:47<00:00, 34.16it/s]



 Classification Report (XLM-RoBERTa + CatBoost):
                         precision    recall  f1-score   support

         appeal_to_fear       0.33      0.00      0.00       449
              bandwagon       0.00      0.00      0.00       236
         cherry_picking       0.46      0.09      0.15       768
                 cliche       0.50      0.00      0.01       694
               euphoria       0.49      0.05      0.10       694
                    fud       0.60      0.06      0.11       576
glittering_generalities       0.71      0.24      0.36       722
        loaded_language       0.69      0.69      0.69      2955
              straw_man       0.00      0.00      0.00       207
           whataboutism       0.00      0.00      0.00       235

              micro avg       0.67      0.31      0.43      7536
              macro avg       0.38      0.11      0.14      7536
           weighted avg       0.54      0.31      0.34      7536
            samples avg       0.37    

In [36]:
# === XLM-RoBERTa + LightGBM ===
model_lgb_xlm = train_model(X_train_xlm, y_train, model_type='lightgbm')

evaluate_model(model_lgb_xlm, X_test_xlm, y_test_true, label_names, description='XLM-RoBERTa + LightGBM')

[LightGBM] [Info] Number of positive: 300, number of negative: 3522
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.138067 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 261120
[LightGBM] [Info] Number of data points in the train set: 3822, number of used features: 1024
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.078493 -> initscore=-2.463002
[LightGBM] [Info] Start training from score -2.463002
[LightGBM] [Info] Number of positive: 157, number of negative: 3665
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.117857 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 261120
[LightGBM] [Info] Number of data points in the train set: 3822, number of used features: 1024
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.041078 -> initscore=-3.150338
[LightGBM] [Info] Start training from score -3.150338
[LightGBM] [